# 03 — Model Comparison
Logistic Regression → Random Forest → XGBoost

In [1]:
import sys, os
# Move to project root so relative paths (data/, outputs/) resolve correctly
# Works whether the kernel starts from notebooks/ or the project root.
_here = os.path.abspath('.')
if os.path.basename(_here) == 'notebooks':
    os.chdir('..')
sys.path.insert(0, os.path.abspath('.'))
import warnings
warnings.filterwarnings('ignore')
import joblib
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from pathlib import Path

from src.models.trainer import train_all_models, save_model
from src.models.evaluator import evaluate_model, compare_models
from src.visualization.plots import plot_roc_curves, plot_pr_curves

FIGURES = Path('outputs/figures')
MODELS  = Path('outputs/models')

## 1. Load Train / Test Split

In [2]:
split = joblib.load('outputs/reports/train_test_split.joblib')
X_train = split['X_train']
y_train = split['y_train']
X_test  = split['X_test']
y_test  = split['y_test']
feature_names = split['feature_names']

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')
print(f'Train churn rate: {pd.Series(y_train).mean():.1%} (post-SMOTE)')
print(f'Test  churn rate: {pd.Series(y_test).mean():.1%}')

X_train: (8278, 37) | X_test: (1409, 37)
Train churn rate: 50.0% (post-SMOTE)
Test  churn rate: 26.5%


## 2. Train All Models

In [3]:
trained = train_all_models(X_train, y_train, X_test, y_test)
print('Trained models:', list(trained.keys()))

Trained models: ['logistic_regression', 'random_forest', 'xgboost']


## 3. Evaluate Each Model

In [4]:
results = {}
for name, model in trained.items():
    metrics = evaluate_model(model, X_test, y_test, name)
    results[name] = metrics
    print(f'{name}: ROC-AUC={metrics["roc_auc"]:.4f}  PR-AUC={metrics["pr_auc"]:.4f}  F2={metrics["f2"]:.4f}')

logistic_regression: ROC-AUC=0.8357  PR-AUC=0.6357  F2=0.6190
random_forest: ROC-AUC=0.8240  PR-AUC=0.6069  F2=0.5946
xgboost: ROC-AUC=0.8151  PR-AUC=0.6106  F2=0.5659


## 4. Comparison Table

In [5]:
comparison = compare_models(results)
print(comparison.to_string())

                     roc_auc  pr_auc      f1      f2  precision  recall  accuracy
model                                                                            
logistic_regression   0.8357  0.6357  0.6132  0.6190     0.6036  0.6230    0.7913
random_forest         0.8240  0.6069  0.5844  0.5946     0.5682  0.6016    0.7729
xgboost               0.8151  0.6106  0.5605  0.5659     0.5518  0.5695    0.7630


## 5. ROC Curves

In [6]:
models_proba = {
    name: model.predict_proba(X_test)[:, 1]
    for name, model in trained.items()
}

path = plot_roc_curves(models_proba, y_test, FIGURES)
img = mpimg.imread(path)
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 6. Precision-Recall Curves

In [7]:
path = plot_pr_curves(models_proba, y_test, FIGURES)
img = mpimg.imread(path)
fig, ax = plt.subplots(figsize=(8, 6), dpi=100)
ax.imshow(img); ax.axis('off')
plt.tight_layout(); plt.show()

## 7. Save Model Artifacts

In [8]:
for name, model in trained.items():
    path = save_model(
        model=model,
        model_name=name,
        metrics=results[name],
        feature_names=feature_names,
        output_dir=MODELS,
    )
    print(f'Saved: {path}')

Saved: C:\Users\mathe\Documents\churn-prediction-pipeline\outputs\models\logistic_regression.joblib
Saved: C:\Users\mathe\Documents\churn-prediction-pipeline\outputs\models\random_forest.joblib
Saved: C:\Users\mathe\Documents\churn-prediction-pipeline\outputs\models\xgboost.joblib


## 8. Which Model to Deploy and Why

**XGBoost is the recommended model** based on the evaluation above.

- **ROC-AUC**: XGBoost consistently leads, indicating superior ability to rank churners ahead of non-churners across all decision thresholds.

- **PR-AUC**: The more honest metric for imbalanced classes. XGBoost's higher Average Precision means fewer false alarms per true churner detected — directly relevant to the cost of wasted retention offers.

- **F2-score**: Weights recall 2× because missing a churner (false negative) costs more than a false alarm. XGBoost's `scale_pos_weight` parameter natively handles the 73/27 class split without resampling, complementing the SMOTE applied to the training data.

- **Logistic Regression** remains valuable as a transparent baseline — its coefficients provide log-odds interpretability for stakeholder presentations where explainability is required over raw performance.

- **Random Forest** is a strong middle ground: more interpretable than XGBoost (native feature importance), more powerful than logistic regression, and more robust to outliers.

For production deployment: **XGBoost + calibration** (notebook 04) to convert raw scores into reliable churn probabilities for risk-based customer segmentation.